In [1]:
import numpy as np
import torch
from dataLoader import *

def resize_tensor(x,size = (7,7)):
    x = x.unsqueeze(0)
    x = torch.nn.functional.interpolate(x, size=size, mode='bilinear')
    return x.squeeze(0)

In [2]:
files_train = sorted(HSI_SATELLITE_DIR.glob("*.npz"))
train_data = []
for file in enumerate(files_train):
    file_name = file[1]
    with np.load(file_name) as npz:
        arr = np.ma.MaskedArray(**npz)
        data = arr.data
        data = np.append(data, [arr.mask[0]], axis=0)
        x = torch.tensor(data, dtype=torch.float32)
        x = resize_tensor(x)
        train_data.append(x)
        
max_width = max(data.shape[1] for data in train_data)
max_height = max(data.shape[2] for data in train_data)
max_size = max_width if max_width > max_height else max_height

train_data = [resize_tensor(data) for data in train_data]

In [3]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

y = gt_train_df[column_names].values
y = scaler.fit_transform(y)
y_binned = np.digitize(y, bins=np.linspace(y.min(), y.max() + 1, 5))
y = torch.tensor(y, dtype=torch.float32)

# fig, axs = plt.subplots(ncols=6, nrows=3, figsize=(10,5))
# for i in range(6):
#     axs[0][i].hist(y[i])
#     axs[1][i].hist(y_train[i])
#     axs[2][i].hist(y_test[i])


# Prepare data and model to be used on the GPU

In [4]:
from torch.utils.data import DataLoader
from neural_networks.Convolutional import CNN2D
from sklearn.model_selection import train_test_split

train_dataset = list(zip(train_data, y))
train, test = train_test_split(train_dataset, train_size=0.8)

# MSI batch size: 64
# MSI neurons: 256
# HSI batch size: 128
# HSI neurons: 128
train_loader = DataLoader(train, batch_size=128, shuffle=True)
test_loader = DataLoader(test, batch_size=len(test), shuffle=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN2D(neurons=256, channels=231).to(device)
criterion = torch.nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = torch.nn.MSELoss()

best_val_loss = float('inf')
best_val_loss_before = best_val_loss

/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


In [5]:
early_stopping_treshold = 128
early_stopping_counter = 0

for epoch in range(5000):
    model.train()
    train_loss= 0.0

    for x,y in train_loader:
        x = x.to(device)
        y = y.to(device)

        output = model(x)
        loss = loss_fn(output, y)
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        train_loss+= loss.item()

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for x,y in test_loader:
            x = x.to(device)
            y = y.to(device)

            output = model(x)
            loss = loss_fn(output, y)

            val_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(test_loader)
    print(f"Epoch: {epoch}, train loss: {avg_train_loss:.4f} val loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        early_stopping_counter = 0

        torch.save(model.state_dict(), "models/nn/CNN2D_HSI.pt")
    else:
        early_stopping_counter += 1
        
        if early_stopping_counter >=early_stopping_treshold:
            break


# best score: 0.5883544087409973 for HSI without warm loading
# best score: 0.5423731803894043 for HSI with warm loading
# best score: 0.5265264511108398 for HSI kernel=1 with warm loading
# 0.5730111002922058 MSI kernel=3 with warm loading
print(best_val_loss)

Epoch: 0, train loss: 1.0209 val loss: 0.9334
Epoch: 1, train loss: 1.0067 val loss: 0.9098
Epoch: 2, train loss: 0.9630 val loss: 0.8586
Epoch: 3, train loss: 0.9325 val loss: 0.8460
Epoch: 4, train loss: 0.9303 val loss: 0.8430
Epoch: 5, train loss: 0.9204 val loss: 0.8296
Epoch: 6, train loss: 0.9009 val loss: 0.7986
Epoch: 7, train loss: 0.8879 val loss: 0.7701
Epoch: 8, train loss: 0.8631 val loss: 0.7618
Epoch: 9, train loss: 0.8551 val loss: 0.7563
Epoch: 10, train loss: 0.8421 val loss: 0.7482
Epoch: 11, train loss: 0.8367 val loss: 0.7412
Epoch: 12, train loss: 0.8305 val loss: 0.7332
Epoch: 13, train loss: 0.8337 val loss: 0.7651
Epoch: 14, train loss: 0.8295 val loss: 0.7472
Epoch: 15, train loss: 0.8150 val loss: 0.6985
Epoch: 16, train loss: 0.8178 val loss: 0.7163
Epoch: 17, train loss: 0.7837 val loss: 0.7006
Epoch: 18, train loss: 0.7710 val loss: 0.7009
Epoch: 19, train loss: 0.7696 val loss: 0.7943
Epoch: 20, train loss: 0.8807 val loss: 0.7558
Epoch: 21, train loss: 

# 3D convolutional neural network (Too slow for hsi)

In [6]:
from neural_networks.Convolutional import CNN3D

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNN3D(neurons=256).to(device)
optimiser = torch.optim.Adam(model.parameters(), lr=1e-4)
best_val_loss = float('inf')
best_val_loss_before = best_val_loss 
criterion = torch.nn.CrossEntropyLoss()
loss_fn = torch.nn.MSELoss()
early_stopping_treshold = 64

In [ ]:
# model.load_state_dict(torch.load("models/nn/CNN3D_HSI.pt"))
# optimiser.param_groups[0]['lr'] = 1e-5
early_stopping_counter = 0

for epoch in range(1000):
    model.train()
    train_loss= 0.0

    for x,y in train_loader:
        x = x.to(device)
        y = y.to(device)

        output = model(x)
        loss = loss_fn(output, y)
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        train_loss+= loss.item()

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for x,y in test_loader:
            x = x.to(device)
            y = y.to(device)

            output = model(x)
            loss = loss_fn(output, y)

            val_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(test_loader)
    print(f"Epoch: {epoch}, train loss: {avg_train_loss:.4f} val loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        early_stopping_counter = 0
        
        torch.save(model.state_dict(), "models/nn/CNN3D_MSI.pt")
    else:
        early_stopping_counter += 1
        
        if early_stopping_counter >=early_stopping_treshold:
            if best_val_loss >= best_val_loss_before:
                break

            best_val_loss_before = best_val_loss
            early_stopping_counter = 0

            model.load_state_dict(torch.load("models/nn/CNN3D_MSI.pt"))

            
print(best_val_loss)

In [ ]:
from pandas import DataFrame

model = CNN2D(channels=231, neurons=256, kernel_size=1)
model.load_state_dict(torch.load("models/nn/CNN2D_HSI.pt"))
model.eval()

files_test = sorted(HSI_SATELLITE_TEST_DIR.glob("*.npz"))
test_data = []
for file in enumerate(files_test):
    file_name = file[1]
    with np.load(file_name) as npz:
        arr = np.ma.MaskedArray(**npz)
        data = arr.data
        data = np.append(data, [arr.mask[0]], axis=0)
        x = torch.tensor(data, dtype=torch.float32)
        x = resize_tensor(x)
        test_data.append(x)
        
max_width = max(data.shape[1] for data in test_data)
max_height = max(data.shape[2] for data in test_data)
max_size = max_width if max_width > max_height else max_height

test_data = [resize_tensor(data) for data in test_data]
test_data = torch.stack(test_data) # comment out for 2D Cnn
with torch.no_grad():
    predictions = model(test_data)

predictions = predictions.cpu().numpy()
predictions = scaler.inverse_transform(predictions)

submission = DataFrame(data = predictions, columns = column_names)
submission.to_csv("cnn2dHsi.csv", index_label="sample_index")

# Check models for

In [11]:
from neural_networks.Convolutional import CNN2D
from neural_networks.Recurrent import FusionNN

def is_fusion_model(model):
    return hasattr(model, "encoders")
    
paths = [
    "models/nn/FusionNN.pt",
    "models/nn/CNN2D_HSI.pt",
    "models/nn/CNN2D_kernel1_MSI.pt"
]

models = [
    FusionNN(neurons=256, hidden_size=12000).to(device),
    CNN2D(neurons=256, channels=231).to(device),
    CNN2D(neurons=256, channels=231, kernel_size=1).to(device)
]

column_map = {
    0: 1, 
    1: 1, 
    2: 0,
    3: 1,
    4: 2,
    5: 0
}

predictions = [[],[],[]]

files_test_HSI = sorted(HSI_SATELLITE_TEST_DIR.glob("*.npz"))
files_test_MSI = sorted(MSI_SATELLITE_TEST_DIR.glob("*.npz"))

predictions = []

for i in range(len(models)):
    print(i)
    model = models[i]
    model.load_state_dict(torch.load(paths[i]))
    model.eval()

    model_preds = []

    with torch.no_grad():
        for idx in range(len(files_test_HSI)):

            # --- HSI ---
            with np.load(files_test_HSI[idx]) as npz:
                arr = np.ma.MaskedArray(**npz)
                data = arr.data
                data = np.append(data, [arr.mask[0]], axis=0)

                x_hsi = torch.tensor(data, dtype=torch.float32)
                x_hsi = resize_tensor(x_hsi)
                x_hsi = x_hsi.unsqueeze(0).to(device)

            # --- MSI ---
            with np.load(files_test_MSI[idx]) as npz:
                arr = np.ma.MaskedArray(**npz)
                data = arr.data
                data = np.append(data, [arr.mask[0]], axis=0)

                x_msi = torch.tensor(data, dtype=torch.float32)
                x_msi = resize_tensor(x_msi)
                x_msi = x_msi.unsqueeze(0).to(device)

            # --- MODEL FORWARD ---
            if hasattr(model, "encoders"):  # FusionNN
                output = model([x_msi, x_hsi])
            else:  # CNN2D
                output = model(x_hsi)

            model_preds.append(output.cpu())

    predictions.append(torch.cat(model_preds, dim=0))


0
1
2


In [14]:
final_preds = torch.zeros_like(predictions[0])

for col in range(6):
    model_id = column_map[col]
    final_preds[:, col] = predictions[model_id][:, col]

final_preds = final_preds.cpu().numpy()
final_preds = scaler.inverse_transform(final_preds)

from pandas import DataFrame

submission = DataFrame(
    data=final_preds,
    columns=column_names
)

submission.to_csv("final.csv", index_label="sample_index")